# 04 — X-IIoTID Model Training

**Objective:** Train five classifiers using every training record.

**Input:** Frozen Core19 training and validation datasets from Notebook 03.

**Output:** Five saved pipelines, validation scores, one comparison graph and a JSON report.

**Rule:** The test dataset is not opened. Labels are used for training and evaluation, never as input features.

In [ ]:
from pathlib import Path
from time import perf_counter
import inspect, json, gc, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
    confusion_matrix, precision_score, recall_score, f1_score)

ROOT = Path(r'D:\J26-CS-336-Network-Flow-V2\network_flow_v2')
SPLIT_REPORT = ROOT / 'reports/audits/x_iiotid_v3_split_report.json'
TRAIN_CSV = ROOT / 'data/splits/x_iiotid/x_iiotid_core19_train_v3.csv.gz'
VALID_CSV = ROOT / 'data/splits/x_iiotid/x_iiotid_core19_validation_v3.csv.gz'
MODEL_DIR = ROOT / 'models/x_iiotid_v3'
METRICS_FILE = ROOT / 'reports/metrics/x_iiotid_v3_five_model_validation.csv'
REPORT_FILE = ROOT / 'reports/audits/x_iiotid_v3_model_training.json'
FIGURE_FILE = ROOT / 'reports/figures/x_iiotid_v3_model_f1.png'
SEED = 42
EXPECTED = {'train': 574585, 'validation': 123125, 'test': 123124}
MODELS = ('decision_tree', 'random_forest', 'extra_trees', 'lightgbm', 'adaboost')
CAT = ['Protocol', 'Service', 'Conn_state', 'is_syn_only', 'Is_SYN_ACK',
       'is_pure_ack', 'is_with_payload', 'FIN or RST']
MISSING = {'', '-', '?', 'na', 'n/a', 'nan', 'none', 'null', 'missing', 'unknown', '<missing>'}

for p in (SPLIT_REPORT, TRAIN_CSV, VALID_CSV):
    if not p.is_file():
        raise FileNotFoundError(f'Missing Notebook 03 output: {p}')
report = json.loads(SPLIT_REPORT.read_text(encoding='utf-8'))
FEATURES = report['features']
NUM = [c for c in FEATURES if c not in CAT]
if len(FEATURES) != 19 or len(NUM) != 11 or not set(CAT).issubset(FEATURES):
    raise ValueError('Unexpected Core19 features. Review Notebook 03.')
# Older Notebook 03 reports may omit total_rows_preserved. The three
# frozen partition counts are mandatory; optional totals must agree if present.
saved_counts = report.get('counts')
if saved_counts != EXPECTED:
    raise ValueError(f'Frozen split counts mismatch: saved={saved_counts!r}; expected={EXPECTED!r}')
expected_total = sum(EXPECTED.values())
saved_total = report.get('total_rows_preserved')
if saved_total is not None and saved_total != expected_total:
    raise ValueError(f'Total rows mismatch: saved={saved_total!r}; expected={expected_total:,}')
verified_counts = report.get('verified_export_counts')
if verified_counts is not None and verified_counts != EXPECTED:
    raise ValueError(f'Export verification mismatch: {verified_counts!r} != {EXPECTED!r}')
print(f'Frozen partition counts verified: {expected_total:,} records across all three partitions.')
if saved_total is None:
    print('Older split report: total_rows_preserved field absent; train/validation file sizes will be checked next.')
if report.get('normalized_signature_overlap_groups') != 0:
    raise ValueError('Feature groups cross partitions: stop before training.')
if report.get('test_scored') is True:
    raise ValueError('Test data was previously scored; investigate experiment protocol.')
for folder in (MODEL_DIR, METRICS_FILE.parent, REPORT_FILE.parent, FIGURE_FILE.parent):
    folder.mkdir(parents=True, exist_ok=True)
print('Features:', len(FEATURES), '| Train:', EXPECTED['train'], '| Validation:', EXPECTED['validation'])
print('Test partition will NOT be read.')

## 1. Load training and validation data

Keep all training rows. Check labels and row IDs before fitting.

In [ ]:
fields = ['csv_row_index_0based'] + FEATURES + ['class1', 'class2', 'class3']
train = pd.read_csv(TRAIN_CSV, dtype=str, keep_default_na=False, low_memory=False)
valid = pd.read_csv(VALID_CSV, dtype=str, keep_default_na=False, low_memory=False)
for name, df in (('train', train), ('validation', valid)):
    if list(df.columns) != fields:
        raise ValueError(f'{name}: wrong column order or missing fields')
    if len(df) != EXPECTED[name]:
        raise ValueError(f'{name}: expected {EXPECTED[name]:,} rows; found {len(df):,}')
    if df['csv_row_index_0based'].duplicated().any():
        raise ValueError(f'{name}: repeated row identifier')
    if not df['class3'].isin(['Normal', 'Attack']).all():
        raise ValueError(f'{name}: unexpected class3 labels')
    for label in ('Normal', 'Attack'):
        expected_count = report['class_counts'][name][label]
        actual_count = int((df['class3'] == label).sum())
        if actual_count != expected_count:
            raise ValueError(f'{name}: {label} count mismatch')
if set(train.csv_row_index_0based).intersection(valid.csv_row_index_0based):
    raise ValueError('Training and validation row IDs overlap')

y_train = train['class3'].map({'Normal': 0, 'Attack': 1}).to_numpy(dtype=np.uint8)
y_valid = valid['class3'].map({'Normal': 0, 'Attack': 1}).to_numpy(dtype=np.uint8)
X_train = train[FEATURES].copy()
X_valid = valid[FEATURES].copy()
del train, valid
gc.collect()
print(f'Full training: {len(y_train):,} rows | Validation: {len(y_valid):,} rows')
print('Only the 19 selected features will enter the models.')

## 2. Prepare features

Fill numeric gaps using training medians and encode text categories. The validation set does not fit these steps.

In [ ]:
def clean_features(frame):
    out = frame[FEATURES].copy()
    for col in NUM:
        values = out[col].astype(str).str.strip()
        values = values.mask(values.str.casefold().isin(MISSING))
        numbers = pd.to_numeric(values, errors='coerce').astype(np.float32)
        out[col] = numbers.replace([np.inf, -np.inf], np.nan)
    for col in CAT:
        values = out[col].astype(str).str.strip()
        out[col] = values.mask(values.str.casefold().isin(MISSING), '__MISSING__')
    return out

X_train = clean_features(X_train)
X_valid = clean_features(X_valid)
encoder_args = {'handle_unknown': 'ignore', 'dtype': np.float32}
if 'sparse_output' in inspect.signature(OneHotEncoder).parameters:
    encoder_args['sparse_output'] = True
else:
    encoder_args['sparse'] = True
imputer_args = {'strategy': 'median'}
if 'keep_empty_features' in inspect.signature(SimpleImputer).parameters:
    imputer_args['keep_empty_features'] = True
preprocessor = ColumnTransformer([
    ('numeric', SimpleImputer(**imputer_args), NUM),
    ('categorical', OneHotEncoder(**encoder_args), CAT)
], sparse_threshold=1.0)

t0 = perf_counter()
X_train_enc = preprocessor.fit_transform(X_train)  # Training data ONLY
X_valid_enc = preprocessor.transform(X_valid)
preprocessing_seconds = perf_counter() - t0
if X_train_enc.shape[0] != EXPECTED['train'] or X_valid_enc.shape[0] != EXPECTED['validation']:
    raise ValueError('Preprocessing changed the number of records')
if X_train_enc.shape[1] != X_valid_enc.shape[1]:
    raise ValueError('Train and validation feature dimensions differ')
print('Training matrix:', X_train_enc.shape)
print('Validation matrix:', X_valid_enc.shape)
print('Preprocessing fit only on training data.')
del X_valid_enc  # Validation is transformed by each saved pipeline during prediction.
gc.collect()

## 3. Train all five classifiers

Use the same feature set and validation partition for each model. Save each complete pipeline.

In [ ]:
def build_classifier(name):
    if name == 'decision_tree':
        return DecisionTreeClassifier(max_depth=14, min_samples_leaf=30,
                                      class_weight='balanced', random_state=SEED)
    if name == 'random_forest':
        return RandomForestClassifier(n_estimators=100, max_depth=16,
            min_samples_leaf=20, class_weight='balanced_subsample', random_state=SEED, n_jobs=-1)
    if name == 'extra_trees':
        return ExtraTreesClassifier(n_estimators=100, max_depth=16,
            min_samples_leaf=20, class_weight='balanced', random_state=SEED, n_jobs=-1)
    if name == 'lightgbm':
        try:
            from lightgbm import LGBMClassifier
        except ImportError as error:
            raise ImportError('Install LightGBM in this notebook kernel: %pip install lightgbm') from error
        return LGBMClassifier(n_estimators=160, num_leaves=31, max_depth=-1,
            learning_rate=0.05, min_child_samples=30, random_state=SEED, n_jobs=4, verbosity=-1)
    if name == 'adaboost':
        weak_tree = DecisionTreeClassifier(max_depth=2, min_samples_leaf=30, random_state=SEED)
        kwargs = {'n_estimators': 100, 'learning_rate': 0.5, 'random_state': SEED}
        param = 'estimator' if 'estimator' in inspect.signature(AdaBoostClassifier).parameters else 'base_estimator'
        kwargs[param] = weak_tree
        return AdaBoostClassifier(**kwargs)
    raise ValueError(f'Unknown classifier: {name}')

results = []
for name in MODELS:
    print(f'\nTraining {name} on {len(y_train):,} rows ...', flush=True)
    classifier = build_classifier(name)
    start = perf_counter()
    classifier.fit(X_train_enc, y_train)
    fit_seconds = perf_counter() - start
    pipeline = Pipeline([('preprocessing', preprocessor), ('classifier', classifier)])
    model_file = MODEL_DIR / f'{name}_core19_v3.joblib'
    joblib.dump(pipeline, model_file, compress=3)
    start = perf_counter()
    predictions = pipeline.predict(X_valid)  # Includes preprocessing; no labels supplied.
    predict_seconds = perf_counter() - start
    tn, fp, fn, tp = (int(x) for x in confusion_matrix(y_valid, predictions, labels=[0,1]).ravel())
    metrics = {
        'model': name,
        'train_rows': len(y_train), 'validation_rows': len(y_valid),
        'accuracy': float(accuracy_score(y_valid, predictions)),
        'balanced_accuracy': float(balanced_accuracy_score(y_valid, predictions)),
        'precision_attack': float(precision_score(y_valid, predictions, zero_division=0)),
        'recall_attack': float(recall_score(y_valid, predictions, zero_division=0)),
        'f1_attack': float(f1_score(y_valid, predictions, zero_division=0)),
        'false_positive_rate': float(fp / max(tn + fp, 1)),
        'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
        'fit_seconds': round(fit_seconds, 3),
        'validation_prediction_seconds': round(predict_seconds, 3),
        'prediction_ms_per_flow_batch_mean': round(1000 * predict_seconds / len(y_valid), 5),
        'model_file_mib': round(model_file.stat().st_size / (1024**2), 4),
    }
    results.append(metrics)
    pd.DataFrame(results).to_csv(METRICS_FILE, index=False)  # Save progress after each model.
    print(f"  Attack F1: {metrics['f1_attack']:.4%} | FPR: {metrics['false_positive_rate']:.4%}")
    print('  Saved:', model_file.name)
    del classifier, pipeline, predictions
    gc.collect()

comparison = pd.DataFrame(results)
if len(comparison) != len(MODELS) or set(comparison['model']) != set(MODELS):
    raise RuntimeError('All five models must finish before the final report')
print('\nAll five models trained using the full training partition.')
display(comparison[['model','train_rows','precision_attack','recall_attack',
                    'f1_attack','false_positive_rate','fit_seconds']].round(4))

## 4. Save results

Show one validation comparison graph. Resource benchmarking and ensemble testing follow in Notebooks 05 and 06.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
values = 100 * comparison['f1_attack'].to_numpy()
bars = ax.bar([n.replace('_', ' ').title() for n in comparison['model']], values)
for bar, value in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'{value:.2f}%', ha='center', va='bottom', fontsize=9)
ax.set_ylim(0, 105)
ax.set_ylabel('Attack F1-score (%)')
ax.set_title('X-IIoTID — Five Classifiers (Validation Only)')
ax.grid(axis='y', alpha=0.2)
plt.xticks(rotation=12, ha='right')
fig.tight_layout()
fig.savefig(FIGURE_FILE, dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)

summary = {
    'dataset': 'X-IIoTID', 'version': 'v3', 'target': 'class3',
    'selected_features': FEATURES, 'excluded_labels': ['class1', 'class2', 'class3'],
    'preprocessing': 'Numeric median imputation and categorical one-hot encoding fitted on train only',
    'frozen_manifest_sha256': report['manifest_sha256'],
    'source_reduced_dataset_sha256': report['source_reduced_dataset_sha256'],
    'train_rows': len(y_train), 'validation_rows': len(y_valid),
    'test_rows_held_out': EXPECTED['test'], 'test_evaluated': False,
    'row_sampling': False, 'random_seed': SEED,
    'preprocessing_fit_transform_seconds': round(preprocessing_seconds, 3),
    'software': {'python': platform.python_version(), 'scikit_learn': sklearn.__version__,
                 'pandas': pd.__version__, 'numpy': np.__version__},
    'model_results': results,
    'limits': [
        'Validation results, not a final held-out test score.',
        'Batch prediction mean is not single-flow p95 latency or Raspberry Pi performance.',
        'Feature availability and missingness shortcuts still require checking.',
        'A custom multi-model ensemble has not been created in this notebook.'
    ]
}
REPORT_FILE.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('Saved validation metrics:', METRICS_FILE)
print('Saved summary:', REPORT_FILE)
print('Saved chart:', FIGURE_FILE)
print('Next: 05_model_evaluation.ipynb — comparison and laptop resource tests.')